# Ilerda: entrenar la NNUE amb 8 buckets de sortida (768 -> HIDDEN -> 1, bullet)
Igual que `ilerda-train.ipynb`, pero el fitxer d'exemple de bullet es crea sencer (`examples/ilerda_buckets.rs`) en lloc de modificar `simple.rs`. Accelerator = GPU T4, Internet = On. Substitueix la ruta de les dades (comprova-la amb la cel·la 2). La xarxa resultant (`quantised.bin`, format amb buckets) la llegeix `Nnue.cs` de la branca `feature/nnue-buckets`.

In [ ]:
!nvidia-smi
!curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>&1 | tail -3

In [ ]:
!find /kaggle/input -name '*.txt' | head

In [ ]:
import os
os.environ['PATH'] += ':/root/.cargo/bin'
!git clone --depth 1 https://github.com/jw1912/bullet /kaggle/working/bullet 2>&1 | tail -2
!cd /kaggle/working/bullet && cargo b -r --package bullet-utils 2>&1 | tail -3

In [ ]:
HIDDEN = 256  # 256 o 512
DATA = '/kaggle/input/datasets/srobis/ilerda-mix2/mix2.txt'
!/kaggle/working/bullet/target/release/bullet-utils convert --from text --input {DATA} --output /kaggle/working/data.bullet --threads 4
!/kaggle/working/bullet/target/release/bullet-utils shuffle --input /kaggle/working/data.bullet --mem-used-mb 8000 --output /kaggle/working/shuf.bullet

In [ ]:
from pathlib import Path
RS = r'''// Exemple de bullet per a Ilerda: (768 -> HIDDEN_SIZE)x2 -> 1 amb 8 buckets de sortida per nombre de peces.
// Basat en examples/progression/2_output_buckets.rs de bullet, amb el mateix calendari d'entrenament
// que la xarxa sense buckets (StepLR 0,001 x0,1 cada 7 superbatches, 20 superbatches, WDL 0,75),
// perque l'unic que canvia respecte de la xarxa actual sigui els buckets.
use bullet_lib::{
    game::{inputs::Chess768, outputs::MaterialCount},
    nn::optimiser::AdamW,
    trainer::{
        save::SavedFormat,
        schedule::{TrainingSchedule, TrainingSteps, lr, wdl},
        settings::LocalSettings,
    },
    value::{ValueTrainerBuilder, loader::DirectSequentialDataLoader},
};

const HIDDEN_SIZE: usize = 256;
const NUM_OUTPUT_BUCKETS: usize = 8;
const SCALE: i32 = 400;
const QA: i16 = 255;
const QB: i16 = 64;

fn main() {
    let mut trainer = ValueTrainerBuilder::default()
        .dual_perspective()
        .optimiser(AdamW)
        .inputs(Chess768)
        .output_buckets(MaterialCount::<NUM_OUTPUT_BUCKETS>)
        .save_format(&[
            SavedFormat::id("l0w").round().quantise::<i16>(QA),
            SavedFormat::id("l0b").round().quantise::<i16>(QA),
            // .transpose(): pesos de sortida [bucket][2N], contigus per bucket (el que llegeix Nnue.cs)
            SavedFormat::id("l1w").round().quantise::<i16>(QB).transpose(),
            SavedFormat::id("l1b").round().quantise::<i16>(QA * QB),
        ])
        .loss_fn(|output, target| output.sigmoid().squared_error(target))
        .build(|builder, stm_inputs, ntm_inputs, output_buckets| {
            let l0 = builder.new_affine("l0", 768, HIDDEN_SIZE);
            let l1 = builder.new_affine("l1", 2 * HIDDEN_SIZE, NUM_OUTPUT_BUCKETS);
            let stm_hidden = l0.forward(stm_inputs).screlu();
            let ntm_hidden = l0.forward(ntm_inputs).screlu();
            l1.forward(stm_hidden.concat(ntm_hidden)).select(output_buckets)
        });

    let schedule = TrainingSchedule {
        net_id: "ilerda".to_string(),
        eval_scale: SCALE as f32,
        steps: TrainingSteps { batch_size: 16_384, batches_per_superbatch: 6104, start_superbatch: 1, end_superbatch: 20 },
        wdl_scheduler: wdl::ConstantWDL { value: 0.75 },
        lr_scheduler: lr::StepLR { start: 0.001, gamma: 0.1, step: 7 },
        save_rate: 5,
    };

    let settings = LocalSettings { threads: 4, test_set: None, output_directory: "/kaggle/working/checkpoints", batch_queue_size: 64 };
    let data_loader = DirectSequentialDataLoader::new(&["/kaggle/working/shuf.bullet"]);

    trainer.run(&schedule, &settings, &data_loader);
}
'''
Path('/kaggle/working/bullet/examples/ilerda_buckets.rs').write_text(RS)


In [ ]:
!cd /kaggle/working/bullet && export PATH=$PATH:/root/.cargo/bin CUDA_PATH=/usr/local/cuda-13.0 RUSTFLAGS='-L /usr/local/nvidia/lib64' && cargo r -r --example ilerda_buckets --features cuda 2>&1 | tail -n 60

In [ ]:
OUT = f'/kaggle/working/ilerda-{HIDDEN}-b8.bin'
!cp /kaggle/working/checkpoints/ilerda-20/quantised.bin {OUT}
!ls -l {OUT}